# 03 – Kiểm định χ² và độ lệch lớn nhất

**Mục tiêu**
1. Tính χ² theo **công thức (5)** của đề bài cho Poisson và Gauss (mục 2.3 bước 5).
2. Hiểu **vì sao** χ² công thức (5) có thể lớn bất thường – phân tích đóng góp từng bin.
3. Kiểm định **χ² Pearson** với gộp bin – cách làm chuẩn trong thống kê.
4. Đánh giá **độ lệch lớn nhất** khi có 100 cơ hội (mục 2.3 bước 4).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from du_lieu import doc_so_lieu, TEN_BO, T_NGAN, T_DAI   # module đọc số liệu (xem notebook 01)

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
data, N100 = doc_so_lieu()
data.head()

,S1,S2,S3
lan_do,,,
1,10,40,126
2,9,26,115
3,7,33,109
4,9,34,128
5,5,37,134


## 1. χ² theo công thức (5)

$$\chi^2=\sum_k\frac{(O_k-E_k)^2}{\sigma_k^2},\qquad \sigma_k=\sqrt{O_k}$$

Vì chia cho $O_k$, bin có $O_k=0$ phải **bỏ qua** (chia cho 0). Bậc tự do $\nu$ = số bin dùng − 1 (đã ước lượng µ từ dữ liệu).

In [2]:
def bang_tan_suat(x):
    k = np.arange(x.min(), x.max() + 1)
    return k, np.bincount(x)[k]

def chi2_cong_thuc_5(O, E):
    m = O > 0
    chi2 = np.sum((O[m] - E[m]) ** 2 / O[m])
    nu = m.sum() - 1
    return chi2, nu, stats.chi2.sf(chi2, nu)

ket_qua = []
for bo in data:
    x = data[bo].to_numpy(); n, mu = len(x), x.mean()
    k, O = bang_tan_suat(x)
    E_P = n * stats.poisson.pmf(k, mu)
    E_G = n * stats.norm.pdf(k, mu, np.sqrt(mu))       # Gauss tính tại số nguyên k (bin rộng 1)
    cP, nu, pP = chi2_cong_thuc_5(O, E_P)
    cG, _, pG = chi2_cong_thuc_5(O, E_G)
    ket_qua.append([bo, cP, cG, nu, cP / nu, pP, pG, "Poisson" if cP <= cG else "Gauss"])
pd.DataFrame(ket_qua, columns=["Bộ", "χ² Poisson", "χ² Gauss", "ν", "χ²/ν (P)", "p (P)", "p (G)", "Tốt hơn"]).round(3)

,Bộ,χ² Poisson,χ² Gauss,ν,χ²/ν (P),p (P),p (G),Tốt hơn
0,S1,28.252,35.548,10,2.825,0.002,0.000,Poisson
1,S2,43.092,44.563,27,1.596,0.026,0.018,Poisson
2,S3,34.111,33.448,40,0.853,0.732,0.758,Gauss


χ²/ν của Bộ 1 ≈ 2,8 và p rất nhỏ – có phải số liệu **không** tuân theo Poisson? Hãy xem từng bin đóng góp bao nhiêu.

## 2. Phân tích đóng góp từng bin

In [3]:
x = data.S1.to_numpy(); n, mu = len(x), x.mean()
k, O = bang_tan_suat(x)
E = n * stats.poisson.pmf(k, mu)
bang = pd.DataFrame({"k": k, "O": O, "E": E, "đóng góp χ²": np.where(O > 0, (O - E) ** 2 / np.maximum(O, 1), 0)})
bang["% tổng"] = 100 * bang["đóng góp χ²"] / bang["đóng góp χ²"].sum()
bang.round(2).sort_values("đóng góp χ²", ascending=False).head(5)

,k,O,E,đóng góp χ²,% tổng
7,8,2,8.56,21.54,76.26
5,6,21,15.74,1.32,4.66
8,9,8,5.25,0.94,3.34
11,12,2,0.67,0.89,3.13
9,10,5,2.90,0.88,3.12


👉 Riêng bin **k = 8** (O = 2, kỳ vọng E ≈ 8,6) đóng góp ~75% tổng χ². Nguyên nhân: công thức (5) dùng $\sigma=\sqrt{O}$ –
khi $O$ **nhỏ do thăng giáng xuống**, mẫu số nhỏ theo → χ² bị phóng đại. (Nếu dùng $\sigma=\sqrt{E}$ thì đóng góp chỉ còn
$(2-8{,}6)^2/8{,}6\approx 5$.)

Cách làm chuẩn: dùng **χ² Pearson** $\sum (O-E)^2/E$ và **gộp bin** sao cho mọi $E\ge 5$.

## 3. χ² Pearson với gộp bin

💡 **Kỹ thuật:** dùng hàm phân bố tích lũy `cdf` để tính xác suất của một nhóm bin $(a, b]$: $P = F(b) - F(a)$;
hai đuôi được gộp vào nhóm đầu/cuối để tổng xác suất = 1.

In [4]:
def chi2_pearson(x, cdf, E_min=5):
    n = len(x)
    # 1) tìm ranh giới: tích lũy từng giá trị nguyên cho tới khi E ≥ E_min
    bien, truoc = [], -np.inf
    for k in range(x.min(), x.max() + 1):
        if n * (cdf(k) - (cdf(truoc) if np.isfinite(truoc) else 0)) >= E_min:
            bien.append(k); truoc = k
    while bien and n * (1 - cdf(bien[-1])) < E_min:   # đuôi phải chưa đủ → gộp vào nhóm trước
        bien.pop()
    canh = [-np.inf] + bien + [np.inf]
    # 2) O và E của từng nhóm
    O = np.array([((x > a) & (x <= b)).sum() for a, b in zip(canh[:-1], canh[1:])])
    F = np.array([0] + [cdf(b) for b in bien] + [1])
    E = n * np.diff(F)
    chi2 = ((O - E) ** 2 / E).sum()
    nu = len(O) - 2                   # trừ 1 (tổng cố định) và 1 (ước lượng µ)
    return chi2, nu, stats.chi2.sf(chi2, nu), O, E

rows = []
for bo in data:
    x = data[bo].to_numpy(); mu = x.mean()
    cP, nuP, pP, O, E = chi2_pearson(x, lambda v: stats.poisson.cdf(v, mu))
    cG, nuG, pG, *_ = chi2_pearson(x, lambda v: stats.norm.cdf(v + 0.5, mu, np.sqrt(mu)))  # hiệu chỉnh liên tục ±0,5
    rows.append([bo, len(O), cP, nuP, pP, cG, nuG, pG])
pd.DataFrame(rows, columns=["Bộ", "số nhóm", "χ² P", "ν P", "p P", "χ² G", "ν G", "p G"]).round(3)

,Bộ,số nhóm,χ² P,ν P,p P,χ² G,ν G,p G
0,S1,9,11.065,7,0.136,10.476,6,0.106
1,S2,15,17.616,13,0.173,18.413,13,0.142
2,S3,15,10.200,13,0.677,16.877,14,0.263


Sau khi gộp bin, p-value của Poisson đều > 0,05 → **không có cơ sở bác bỏ** giả thuyết Poisson ở cả ba mức.

> 💡 Gauss được rời rạc hóa với **hiệu chỉnh liên tục**: bin nguyên $k$ ứng với khoảng $(k-0{,}5;\ k+0{,}5]$.

## 4. Độ lệch lớn nhất – “có 100 cơ hội”

Với Bộ 1, giá trị xa trung bình nhất là bao nhiêu σ? Xác suất **một** lần đo lệch như vậy là
$p = P(X\ge x_{max})$ = `stats.poisson.sf(x_max - 1, mu)` (lưu ý `sf(k)` = $P(X>k)$).
Xác suất **ít nhất một lần** trong $n=100$ lần: $1-(1-p)^{n}$.

In [5]:
x = data.S1.to_numpy(); n, mu = len(x), x.mean()
x_max = x.max()
p1 = stats.poisson.sf(x_max - 1, mu)          # P(X ≥ x_max)
p100 = 1 - (1 - p1) ** n
print(f"x̄ = {mu:.2f}, σ = √x̄ = {np.sqrt(mu):.2f}")
print(f"x_max = {x_max} → lệch {(x_max - mu) / np.sqrt(mu):.2f}σ")
print(f"P(một lần đo ≥ {x_max})      = {p1:.4f}  ({p1:.1%})")
print(f"P(≥ 1 lần trong {n} lần đo)  = {p100:.3f}  ({p100:.0%})")
print(f"Số lần kỳ vọng               = {n * p1:.2f}")

x̄ = 5.52, σ = √x̄ = 2.35
x_max = 12 → lệch 2.76σ
P(một lần đo ≥ 12)      = 0.0113  (1.1%)
P(≥ 1 lần trong 100 lần đo)  = 0.678  (68%)
Số lần kỳ vọng               = 1.13


**Kết luận:** riêng một lần đo, 1,1% có vẻ hiếm; nhưng với 100 cơ hội, xác suất gặp ít nhất một lần là ~68% → hoàn toàn bình thường,
hệ đếm hoạt động tốt. Đây là bài học về **hiệu ứng so sánh nhiều lần** (look-elsewhere effect).

Kiểm tra lại bằng mô phỏng (xem thêm notebook 04):

In [6]:
rng = np.random.default_rng(1)
gia_lap = rng.poisson(mu, size=(100_000, n))          # 100 000 thí nghiệm ảo × 100 lần đo
ty_le = (gia_lap.max(axis=1) >= x_max).mean()
print(f"Tỉ lệ thí nghiệm ảo có max ≥ {x_max}: {ty_le:.3f}  (lý thuyết {p100:.3f})")

Tỉ lệ thí nghiệm ảo có max ≥ 12: 0.680  (lý thuyết 0.678)


## 📝 Bài tập
1. Tính lại χ² công thức (5) nhưng dùng $\sigma_k=\sqrt{E_k}$ (tức Pearson **không** gộp bin). So sánh với hai cách trên.
2. Lặp lại phân tích độ lệch lớn nhất cho Bộ 3 (lệch về phía **dưới**: dùng `stats.poisson.cdf(x_min, mu)`).
3. Dùng mô phỏng: sinh 10 000 mẫu Poisson 100 lần đo với µ = 5,52, tính χ² công thức (5) cho mỗi mẫu. Phân bố của nó có giống $\chi^2(\nu)$ không? Từ đó kết luận về độ tin cậy của công thức (5).